[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/21_gradient_clipping_solution.ipynb)

# 🟢 Solution: Gradient Norm Clipping

*Training · Easy*

Reference implementation. Try it yourself in `21_gradient_clipping.ipynb` first.

---
Implement **global-norm gradient clipping**.

$$g \leftarrow g \cdot \min\left(1, \frac{\text{max\_norm}}{\|g\|_2 + \epsilon}\right)
\qquad \|g\|_2 = \sqrt{\sum_{\text{all leaves}} \sum_i g_i^2}$$

### Signature
```python
def clip_grad_norm(grads, max_norm):
    ...  # -> (clipped_grads, total_norm)
```

### Rules
- The norm is **global** — one number across the entire gradient pytree, not
  per-tensor
- Use `1e-6` in the denominator, matching the reference
- Only scale when the coefficient is `< 1`
- Do not use `optax.clip_by_global_norm`

### Why global and not per-tensor
Scaling every leaf by the *same* coefficient preserves the **direction** of the
update — you shorten the step without rotating it. Per-tensor clipping rescales
each tensor independently, which changes the relative sizes of the layer
updates and therefore points you somewhere else entirely. Global-norm is what
every large-model training script uses, usually at `max_norm=1.0`.

### What it is actually for
Clipping is a guard against loss spikes. A single bad batch — a long sequence,
a degenerate example — can produce a gradient orders of magnitude larger than
usual, and one such step is enough to knock a large model into a region it
never recovers from. Clipping bounds the damage of that step.

How often it actually fires depends on the run: with `max_norm=1.0` it is
common for a large fraction of early pretraining steps to be clipped, tailing
off as training settles. So treat it as an always-on safety rail whose binding
rate you should watch — a clip rate near 100% late in training usually means
the threshold is too low, not that the model is unstable.

### ⚠️ JAX-forced signature change
PyTorch takes an iterable of parameters, reads `p.grad`, and mutates it via
`p.grad.mul_(coef)`. In JAX gradients are a plain pytree returned by
`jax.grad`, and arrays are immutable — so this takes the **gradient pytree**
and **returns** the clipped one, alongside the norm.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def clip_grad_norm(grads, max_norm):
    # One norm across every leaf — this is what makes it "global".
    total_norm = jnp.sqrt(
        sum(jnp.sum(g ** 2) for g in jax.tree.leaves(grads))
    )

    clip_coef = max_norm / (total_norm + 1e-6)
    # Never scale UP: a gradient already inside the ball is left alone.
    clip_coef = jnp.minimum(clip_coef, 1.0)

    clipped = jax.tree.map(lambda g: g * clip_coef, grads)
    return clipped, total_norm

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

grads = {"w": jnp.array([3.0, 4.0]), "b": jnp.array(0.0)}   # norm = 5
clipped, norm = clip_grad_norm(grads, max_norm=1.0)

print("norm before:", float(norm))
print("clipped    :", clipped["w"], "-> norm", float(jnp.linalg.norm(clipped["w"])))
print("direction preserved:", clipped["w"] / jnp.linalg.norm(clipped["w"]))

small = {"w": jnp.array([0.1, 0.0])}
out, n = clip_grad_norm(small, max_norm=1.0)
print("\nsmall gradient untouched:", out["w"], "(norm", float(n), ")")

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("gradient_clipping")